# MSD RecSys — pipeline notebook

End-to-end driver for the Million Song Dataset two-stage recommender.
The library lives in `msd_recsys/`; this notebook just stitches the stages together
and prints diagnostics between them.

**Stages**
1. Setup (install, mount Drive, set paths)
2. Load + filter + split data
3. Stage 1 — retrieval (ALS + semantic IDs -> hybrid pool)
4. Stage 2 — features + ranker
5. Evaluation (overall + bucketed)
6. Submission


## 1. Setup


In [1]:
# --- Colab-only: clone fresh + install. Safe to re-run. ---
import os
os.chdir('/content')                    # so we're not inside the dir we're about to delete
!rm -rf ecs172-recommender-systems
!git clone https://github.com/andrewscoding2018/ecs172-recommender-systems.git
%cd ecs172-recommender-systems
!pip install -q -e ./final_project

# --- Mount Drive for checkpoints (no-op if already mounted) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not in Colab — running locally")

Cloning into 'ecs172-recommender-systems'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 128 (delta 62), reused 91 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 21.15 MiB | 33.52 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/ecs172-recommender-systems
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for msd_recsys (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip uninstall -y implicit
!pip install --no-cache-dir --no-binary=implicit implicit

Found existing installation: implicit 0.7.3
Uninstalling implicit-0.7.3:
  Successfully uninstalled implicit-0.7.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 11.7 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.3-cp312-cp312-linux_x86_64.whl size=894288 sha256=1bc34c18db264abc7e63141405df132b7f4f50b7731e7584b2ef19a928a815b0
  Stored in directory: /tmp/pip-ephem-wheel-cache-dwzlgyhw/wheels/4f/35/c0/f81ac5a663e9facb00a548c068a3f558d9a32b9ea107297831
Successfully built implicit


In [3]:
from implicit.gpu import HAS_CUDA
print(f"HAS_CUDA: {HAS_CUDA}")

HAS_CUDA: False


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

from msd_recsys import data, retrieval, features, ranker, eval as ev, diagnostics as diag
from msd_recsys.checkpoint import checkpoint, set_checkpoint_dir, list_checkpoints

# Adjust these for your environment.
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/ECS 172')
    CKPT_DIR = Path('/content/drive/MyDrive/ECS 172/msd_checkpoints')
else:
    DATA_DIR = Path('./data')
    CKPT_DIR = Path('./checkpoints')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
set_checkpoint_dir(CKPT_DIR)
print(f"DATA_DIR = {DATA_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")
print(f"existing checkpoints: {list_checkpoints()}")


DATA_DIR = /content/drive/MyDrive/ECS 172
CKPT_DIR = /content/drive/MyDrive/ECS 172/msd_checkpoints
existing checkpoints: [('als_f64_r0.01_i30', 218.945811), ('als_topk_v_1000', 5588.44405), ('als_topk_v_4000', 2860.469702), ('interactions_filtered_s50_u20', 737.875302), ('interactions_raw_v1', 944.950125), ('metadata_v1', 155.779752), ('sid_metadata_v1', 8.445439), ('sid_topk_v_1000', 5616.515288), ('sid_topk_v_4000', 2871.475735), ('split_n5', 737.875737), ('ui_matrix_v1', 324.223309)]


## 2. Load + filter + split


In [5]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/ECS 172')

for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name:50s}  {p.stat().st_size / 1e6:>10.1f} MB")

  Andrew_Kuang_ECS_172_Assignment_1.gdoc                     0.0 MB
  Assignment 3 Report.gdoc                                   0.0 MB
  ECS172 Final Presentation.gslides                          0.0 MB
  MSDChallengeGettingstarted.pdf                             0.2 MB
  kaggle_songs.txt                                           9.9 MB
  kaggle_users.txt                                           4.5 MB
  kaggle_visible_evaluation_triplets.txt                    90.0 MB
  kaggle_visible_evaluation_triplets.zip                    18.4 MB
  msd_checkpoints                                            0.0 MB
  msdchallenge.zip                                          31.2 MB
  taste_profile_song_to_tracks.txt                          14.6 MB
  taste_profile_song_to_tracks.txt.zip                       6.3 MB
  track_metadata.db                                        746.2 MB
  train_triplets (1).txt                                  3001.7 MB
  train_triplets.txt                            

In [6]:
# --- Big files: training interactions + metadata DB ---
interactions = checkpoint(
    "interactions_raw_v1",
    lambda: data.load_triplets(DATA_DIR / "train_triplets.txt"),
)
metadata = checkpoint(
    "metadata_v1",
    lambda: data.load_track_metadata(DATA_DIR / "track_metadata.db"),
)

# --- Canonical Kaggle files (small, no need to checkpoint) ---
# kaggle_songs.txt:                     canonical catalog (~386K song IDs)
# kaggle_users.txt:                     canonical user list (~110K user IDs we must predict for)
# kaggle_visible_evaluation_triplets:   visible half of the eval users' history (~1.45M rows)
canonical_songs = data.load_song_id_list(DATA_DIR / "kaggle_songs.txt")
canonical_users = data.load_user_list(DATA_DIR / "kaggle_users.txt")
visible_eval    = data.load_triplets(DATA_DIR / "kaggle_visible_evaluation_triplets.txt")

# taste_profile_song_to_tracks.txt is not strictly required — triplets and
# metadata.db both key on song_id, so we can join directly. Load it only if you
# have it (e.g., for cross-checking against per-track MSD audio files later).
song_to_track = None
if (DATA_DIR / "taste_profile_song_to_tracks.txt").exists():
    song_to_track = data.load_song_to_track(DATA_DIR / "taste_profile_song_to_tracks.txt")

print(f"interactions (train):    {interactions.shape}")
print(f"metadata:                {metadata.shape}")
print(f"canonical songs:         {len(canonical_songs):,}")
print(f"canonical users:         {len(canonical_users):,}")
print(f"visible eval triplets:   {visible_eval.shape}")
if song_to_track is not None:
    print(f"song-to-track mapping:   {song_to_track.shape}")
diag.diag_data(interactions, items_catalog=metadata)


[ckpt] loading interactions_raw_v1  (945.0 MB)
[ckpt] loading metadata_v1  (155.8 MB)
interactions (train):    (48373586, 3)
metadata:                (1000000, 10)
canonical songs:         386,213
canonical users:         110,000
visible eval triplets:   (1450933, 3)
song-to-track mapping:   (384541, 2)
[diag] interactions: 48,373,586 rows, 1,019,318 users, 384,546 items
[diag] catalog: 1,000,000 items, 61.5% are cold (no train interactions)
       -> any cold-item recall comes from the content/semantic-ID route.
[diag] user history: median=27, p90=105, max=4400
[diag] song popularity: median=13, mean=125.8, max=110479
       -> long-tailed; aggressive item filtering will reshape the distribution.


In [7]:
# --- Aggressive filter: drop tail items and low-activity users ---
MIN_SONG_LISTENS = 50
MIN_USER_LISTENS = 20

filtered = checkpoint(
    f"interactions_filtered_s{MIN_SONG_LISTENS}_u{MIN_USER_LISTENS}",
    lambda: data.filter_interactions(
        interactions,
        min_song_listens=MIN_SONG_LISTENS,
        min_user_listens=MIN_USER_LISTENS,
    ),
)
diag.diag_filter(interactions, filtered)


[ckpt] loading interactions_filtered_s50_u20  (737.9 MB)
[diag] after filter:
  rows:    48,373,586 ->   39,327,173  (81.3% kept)
  users:    1,019,318 ->      624,994  (61.3% kept)
  items:      384,546 ->       91,296  (23.7% kept)
       -> tail items dropped; ALS now operates on a denser, more reliable signal.


In [8]:
# --- Hold-out split mirroring the MSD challenge format ---
KEEP_LAST_N = 5
train_inner, valid = checkpoint(
    f"split_n{KEEP_LAST_N}",
    lambda: data.holdout_split(filtered, n_per_user=KEEP_LAST_N),
)
print(f"train_inner: {len(train_inner):,} rows, {train_inner.user_id.nunique():,} users")
print(f"valid      : {len(valid):,} rows,    {valid.user_id.nunique():,} users")


[ckpt] loading split_n5  (737.9 MB)
train_inner: 36,202,203 rows, 624,994 users
valid      : 3,124,970 rows,    624,994 users


In [9]:
# --- Build sparse user-item matrix for ALS ---
ui_matrix, user_to_ix, item_to_ix = checkpoint(
    "ui_matrix_v1",
    lambda: data.build_user_item_matrix(train_inner, confidence_alpha=40.0),
)

print(f"user-item matrix: {ui_matrix.shape}, nnz={ui_matrix.nnz:,}")


[ckpt] loading ui_matrix_v1  (324.2 MB)
user-item matrix: (624994, 90862), nnz=36,202,203


## 3. Stage 1 — Retrieval


In [10]:
# --- ALS via implicit (GPU-accelerated on A100 if available) ---
als = checkpoint(
    "als_f64_r0.01_i30",
    lambda: retrieval.ALSRetriever(factors=64, regularization=0.01, iterations=30).fit(
        ui_matrix, user_to_ix, item_to_ix,
    ),
)

# Optional sanity check — pass a few well-known song titles in if you have them.
# Build a title lookup from metadata for prettier diagnostic output.
title_by_song = dict(zip(metadata.song_id, metadata.title))
diag.diag_als(als, item_titles=title_by_song)


[ckpt] loading als_f64_r0.01_i30  (218.9 MB)
[diag] ALS similar-item check:
  The Cove:
    sim=0.989  Questions
    sim=0.986  Wrong Turn
    sim=0.985  Rainbow
    sim=0.985  Symbol In My Driveway
    sim=0.984  Supposed To Be
  Nothing from Nothing:
    sim=0.921  Ride Captain Ride
    sim=0.914  Soul Finger
    sim=0.910  Groovin'
    sim=0.909  Will It Go Round In Circles
    sim=0.908  Fiera Callada
  Entre Dos Aguas:
    sim=0.949  Almoraima
    sim=0.947  Cepa Andaluza
    sim=0.935  Mediterranean Sundance / Rio Ancho
    sim=0.931  Guajiras De Lucia
    sim=0.921  Chan Chan (Live)
  -> if similar songs are same artist / same era / same genre, ALS is learning real structure.


In [11]:
# --- Semantic-ID retriever (metadata-only for now; revisit if adding audio) ---
# Filter metadata to the items that appear in our filtered training set so the
# semantic-ID space matches what we recommend over.
items_in_train = set(train_inner.song_id.unique())
metadata_in_train = metadata[metadata.song_id.isin(items_in_train)].copy()

sid = checkpoint(
    "sid_metadata_v1",
    lambda: retrieval.SemanticIDRetriever().fit(metadata_in_train),
)
diag.diag_semantic_codes(sid, item_titles=title_by_song, n_show=8)


[ckpt] loading sid_metadata_v1  (8.4 MB)
[diag] Semantic-ID code distribution:
  Rock-N-Rule                              codes=(49, 133, 164, 0, 0)
  Raspberry Beret (LP Version)             codes=(75, 231, 73, 1, 1)
  All of the same blood                    codes=(90, 16, 202, 0, 2)
  Scream                                   codes=(104, 0, 126, 0, 3)
  Dancing In The Dark                      codes=(175, 122, 233, 0, 4)
  Remember (Walking In The Sand)           codes=(177, 62, 84, 2, 5)
  Lucy Fears the Morning Star              codes=(21, 216, 142, 0, 6)
  Get Along (Feat: Pace Won) (Instrumental codes=(4, 198, 214, 0, 7)
  -> items with overlapping codes should share metadata (decade, artist, popularity tier).


In [12]:
# === CELL A: setup (cheap, instant) ===
TOPK_ALS = 4000
TOPK_SID = 4000

valid_users = sorted(valid.user_id.unique())
valid_users = [u for u in valid_users if u in user_to_ix]
valid_user_indices = np.array([user_to_ix[u] for u in valid_users])
hist_by_user = data.histories_from_df(train_inner)
valid_histories = [hist_by_user.get(u, []) for u in valid_users]
valid_truth = valid.groupby("song_id" if False else "user_id")["song_id"].apply(set).to_dict()

print(f"valid users to retrieve for: {len(valid_users):,}")

valid users to retrieve for: 624,994


In [21]:
# In CELL A, after building valid_users:
SAMPLE_N = None  # set to None for full run
if SAMPLE_N and len(valid_users) > SAMPLE_N:
    rng = np.random.default_rng(42)
    keep = rng.choice(len(valid_users), SAMPLE_N, replace=False)
    valid_users = [valid_users[i] for i in keep]
    valid_user_indices = valid_user_indices[keep]
    valid_histories = [valid_histories[i] for i in keep]
    print(f"sampled down to {len(valid_users):,} users for fast iteration")

In [22]:
# === CELL B: ALS retrieval (potentially slow, then cached) ===
import time
t = time.time()
als_ids, als_sc = checkpoint(
    f"als_topk_v_{TOPK_ALS}",
    lambda: als.recommend_batch(ui_matrix, top_k=TOPK_ALS, user_indices=valid_user_indices),
)
print(f"ALS retrieval: {time.time() - t:.1f}s  ({als_ids.shape})")

[ckpt] loading als_topk_v_4000  (2860.5 MB)
ALS retrieval: 16.2s  ((80000, 4000))


In [23]:
# === CELL C: Semantic-ID retrieval (the other potentially slow op) ===
import time
t = time.time()
sid_ids, sid_sc = checkpoint(
    f"sid_topk_v_{TOPK_SID}",
    lambda: sid.recommend_for_histories(
        valid_histories, top_k=TOPK_SID,
        already_owned=[set(h) for h in valid_histories],
    ),
)
print(f"SID retrieval: {time.time() - t:.1f}s  ({sid_ids.shape})")

[ckpt] loading sid_topk_v_4000  (2871.5 MB)
SID retrieval: 17.9s  ((80000, 4000))


In [28]:
# === CELL D: pool building on a sampled slice ===
# print(f"valid_users full size: {len(valid_users):,}  →  sampling {SAMPLE_N:,}")

# rng = np.random.default_rng(42)
# keep = rng.choice(len(valid_users), SAMPLE_N, replace=False)

# Slice every aligned array/list to the same subset
# valid_users_s     = [valid_users[i]     for i in keep]
valid_users_s = valid_users
# valid_histories_s = [valid_histories[i] for i in keep]
valid_histories_s = valid_histories
# als_ids_s, als_sc_s = als_ids[keep], als_sc[keep]
als_ids_s, als_sc_s = als_ids, als_sc
# sid_ids_s, sid_sc_s = sid_ids[keep], sid_sc[keep]
sid_ids_s, sid_sc_s = sid_ids, sid_sc



# Now the original Cell D code, on the smaller arrays:
valid_cands = retrieval.build_candidate_pool(als_ids_s, als_sc_s, sid_ids_s, sid_sc_s)
diag.diag_pool(valid_cands, truth_by_user=valid_truth, user_ids=valid_users_s)

: 

: 

: 

## 4. Stage 2 — Features + ranker


In [ ]:
# --- Build item feature table (one-shot, used by every (user, candidate) row) ---
item_pop = train_inner.groupby("song_id").size()
item_table = features.ItemFeatureTable.build(
    metadata=metadata_in_train,
    item_popularity=item_pop,
    semantic_codes=sid.item_codes,
)
print(f"item feature table: {len(item_table.item_ids):,} items")


item feature table: 91,461 items


In [ ]:
# --- 80/20 split of validation users for ranker fit vs eval ---
rng = np.random.default_rng(42)
perm = rng.permutation(len(valid_users))
split = int(0.8 * len(valid_users))
fit_ix, eval_ix = perm[:split], perm[split:]

def slice_(arr, ixs): 
    return [arr[i] for i in ixs]
fit_users  = slice_(valid_users,     fit_ix)
fit_hist   = slice_(valid_histories, fit_ix)
fit_cands  = slice_(valid_cands,     fit_ix)
eval_users = slice_(valid_users,     eval_ix)
eval_hist  = slice_(valid_histories, eval_ix)
eval_cands = slice_(valid_cands,     eval_ix)

X_fit,  y_fit,  k_fit,  g_fit  = features.build_feature_rows(
    fit_users, fit_hist, fit_cands, item_table, truth_by_user=valid_truth,
)
X_eval, y_eval, k_eval, g_eval = features.build_feature_rows(
    eval_users, eval_hist, eval_cands, item_table, truth_by_user=valid_truth,
)

# Drop zero-sized groups (users whose candidate dicts came back empty)
g_fit_nz  = [g for g in g_fit  if g > 0]
g_eval_nz = [g for g in g_eval if g > 0]
print(f"ranker fit : X={X_fit.shape}, positives={int(y_fit.sum()):,} ({y_fit.mean():.2%})")
print(f"ranker eval: X={X_eval.shape}, positives={int(y_eval.sum()):,} ({y_eval.mean():.2%})")
diag.diag_features(X_fit, y_fit, features.FEATURE_NAMES)


ranker fit : X=(7688039, 11), positives=1,383 (0.02%)
ranker eval: X=(1925947, 11), positives=362 (0.02%)
[diag] feature means by class (discriminative power):
  feature                       positive      negative     ratio
  hist_size_log                   3.4830        3.7656     0.92x
  distinct_artists_log            3.1129        3.2902     0.95x
  avg_artist_familiarity          0.7484        0.7377     1.01x
  pop_log                         8.1052        6.5282     1.24x
  artist_familiarity              0.7967        0.7564     1.05x
  artist_hotttnesss               0.6470        0.5663     1.14x
  decade                       1689.3398     1754.8806     0.96x
  als_score                       0.3261        0.1872     1.74x
  semantic_score                 12.7932       21.7639     0.59x
  shared_artists                  0.4143        0.0248    16.68x
  semantic_match_count            0.9848        0.6723     1.46x
  -> ratios far from 1.0 mean the feature separates the clas

In [ ]:
# --- Train LightGBM lambdarank ---
model = ranker.train_lambdarank(
    X_fit, y_fit, group_sizes=g_fit_nz,
    eval_set=(X_eval, y_eval), eval_group=g_eval_nz,
    n_estimators=300, max_position=500, early_stopping_rounds=20,
)
diag.diag_permutation_importance(model, X_eval, y_eval, features.FEATURE_NAMES, rng=rng)


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[26]	valid_0's map@10: 0.779152	valid_0's map@100: 0.784152	valid_0's map@500: 0.784351


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with fea

[diag] permutation importance (sampled 30K eval rows, AP scoring):
  feature                   importance       std
  shared_artists               0.02462   0.00006
  pop_log                      0.02177   0.00194
  artist_hotttnesss            0.01143   0.00052
  distinct_artists_log         0.00310   0.00252
  semantic_score               0.00112   0.00051
  hist_size_log                0.00104   0.00607
  artist_familiarity           0.00045   0.00105
  avg_artist_familiarity      -0.00077   0.00104
  decade                      -0.00082   0.00145
  semantic_match_count        -0.00108   0.00016
  als_score                   -0.01793   0.00201
  -> drop in AP when feature is shuffled; higher = ranker relies on it more.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with fea

## 5. Evaluation


In [ ]:
# --- Rank candidates -> top-500 per user, compute metrics ---
eval_owned = {u: set(h) for u, h in zip(eval_users, eval_hist)}
ranked = ranker.rank_candidates(model, X_eval, k_eval, top_k=500, exclude_owned=eval_owned)

map500    = ev.map_at_k(ranked,    {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
recall500 = ev.mean_recall_at_k(ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
ndcg500   = ev.mean_ndcg_at_k(ranked,   {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

# Retrieval ceiling on the eval slice
ret_recall = ev.retrieval_recall(
    {u: set(c.keys()) for u, c in zip(eval_users, eval_cands)},
    {u: valid_truth[u] for u in eval_users if u in valid_truth},
)

# Sum-of-scores baseline
baseline_scores = X_eval[:, features.FEATURE_NAMES.index("als_score")] \
                + X_eval[:, features.FEATURE_NAMES.index("semantic_score")]
baseline_ranked: dict = {}
for (u, it), s in zip(k_eval, baseline_scores):
    baseline_ranked.setdefault(u, []).append((it, float(s)))
for u, lst in baseline_ranked.items():
    lst.sort(key=lambda x: -x[1])
    owned = eval_owned.get(u, set())
    baseline_ranked[u] = [it for it, _ in lst if it not in owned][:500]
baseline_map500 = ev.map_at_k(baseline_ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

print(f"MAP@500     : {map500:.4f}")
print(f"Recall@500  : {recall500:.4f}")
print(f"NDCG@500    : {ndcg500:.4f}")
print(f"baseline MAP@500 (als+sid sum): {baseline_map500:.4f}")
diag.diag_results(map_score=map500, retrieval_recall=ret_recall, baseline_map=baseline_map500)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


MAP@500     : 0.0135
Recall@500  : 0.0640
NDCG@500    : 0.0365
baseline MAP@500 (als+sid sum): 0.0001
[diag] putting it together:
  Retrieval ceiling (Recall@pool) : 0.0724
  Achieved MAP@500              : 0.0135
  Capture efficiency              : 18.6%
  Lift over baseline              : 97.45x
  -> if capture efficiency is low, ranker is the bottleneck; if ceiling is low, retrieval is.


In [ ]:
# --- Bucketed evaluation: by user history size ---
user_hist_size = {u: len(h) for u, h in zip(eval_users, eval_hist)}
def history_bucket(n):
    if n < 30:   return "0_light"     # leading underscores for sort order
    if n < 100:  return "1_medium"
    return "2_heavy"
user_buckets = {u: history_bucket(n) for u, n in user_hist_size.items()}

bucketed = ev.metrics_by_bucket(ranked, {u: valid_truth[u] for u in eval_users}, user_buckets, k=500)
print("by user-history bucket:")
print(bucketed.to_string(index=False))


by user-history bucket:
  bucket  users  MAP@500  Recall@500  NDCG@500
 0_light    377 0.020207    0.099204  0.054834
1_medium    462 0.011516    0.050649  0.030525
 2_heavy    161 0.003475    0.019876  0.010456


In [ ]:
# --- Catalog coverage + tail-focused MAP ---
catalog_size = len(item_table.item_ids)
coverage = ev.catalog_coverage(ranked, catalog_size, k=500)
print(f"catalog coverage @500: {coverage:.2%}  ({int(coverage * catalog_size):,} of {catalog_size:,} items recommended to >=1 user)")

# Popularity tiers (head/torso/tail by cumulative listen share)
item_tier = ev.popularity_tiers(item_pop)
tail_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="tail")
torso_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="torso")
head_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="head")
print(f"\nMAP@500 by ground-truth popularity tier:")
print(f"  head : {head_map:.4f}")
print(f"  torso: {torso_map:.4f}")
print(f"  tail : {tail_map:.4f}")
print("  -> the tail number is the 'discovery' signal — does the system help users find long-tail songs they actually played?")


catalog coverage @500: 22.01%  (20,127 of 91,461 items recommended to >=1 user)

MAP@500 by ground-truth popularity tier:
  head : 0.0509
  torso: 0.0028
  tail : 0.0024
  -> the tail number is the 'discovery' signal — does the system help users find long-tail songs they actually played?


In [ ]:
truth_items = set()
for u in valid_users_s:
    truth_items |= valid_truth.get(u, set())

train_items = set(train_inner.song_id.unique())
sid_items   = set(sid.all_items)

print(f"unique held-out items     : {len(truth_items):,}")
print(f"  also scorable by ALS    : {len(truth_items & train_items):,}  ({len(truth_items & train_items)/len(truth_items):.1%})")
print(f"  also scorable by SID    : {len(truth_items & sid_items):,}  ({len(truth_items & sid_items)/len(truth_items):.1%})")
print(f"  unreachable by EITHER   : {len(truth_items - train_items - sid_items):,}  ({len(truth_items - train_items - sid_items)/len(truth_items):.1%})")

unique held-out items     : 8,476
  also scorable by ALS    : 8,205  (96.8%)
  also scorable by SID    : 8,205  (96.8%)
  unreachable by EITHER   : 271  (3.2%)


In [ ]:
hist_sizes = pd.Series([len(h) for h in valid_histories_s])
print(f"valid history sizes (after last-5 holdout):")
print(f"  min/p25/median/p75/max: {hist_sizes.min()} / {hist_sizes.quantile(.25):.0f} / {hist_sizes.median():.0f} / {hist_sizes.quantile(.75):.0f} / {hist_sizes.max()}")
print(f"  users with ≤5 listens : {(hist_sizes <= 5).sum():,} ({(hist_sizes <= 5).mean():.0%})")

valid history sizes (after last-5 holdout):
  min/p25/median/p75/max: 15 / 23 / 37 / 69 / 834
  users with ≤5 listens : 0 (0%)


In [ ]:
pop = train_inner.groupby("song_id").size().sort_values(ascending=False)
pop_top = pop.head(2000).index.tolist()  # match your ~1922 pool size

# How many held-out hits would top-2000-by-popularity catch?
pop_set = set(pop_top)
hits = total = 0
for u, t in valid_truth.items():
    if u not in set(valid_users_s):
        continue
    if not t:
        continue
    hits += len(t & pop_set)
    total += len(t)
print(f"popularity-baseline recall@2000: {hits/total:.4f}")

popularity-baseline recall@2000: 0.1555


## 6. Submission


In [ ]:
# --- Retrain Stage 1 on full filtered data + rank for test users ---
# Skeleton — adapt once you've confirmed which hidden test file to evaluate against.

# 1. Rebuild ui_matrix on `filtered` (not train_inner), refit ALS, refit SID.
# 2. Load test user_ids from kaggle_users.txt or the canonical hidden file.
# 3. For each test user, retrieve TOPK_ALS + TOPK_SID, build features, rank to top-500.
# 4. Format CSV per the MSD challenge format and validate.

# Placeholder — uncomment and adapt:
# test_users = data.load_user_list(DATA_DIR / "kaggle_users.txt")
# ... retrieve + rank ...
# submission.to_csv('submission.csv', index=False)
print("submission cell is a skeleton — fill in once test format is confirmed.")


submission cell is a skeleton — fill in once test format is confirmed.
